### Problem 1

In [33]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from scipy.spatial.distance import cosine

In [34]:
data=pd.read_csv(r'ml-100k/u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])
data.head()

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [35]:
data.shape

(100000, 4)

In [36]:
data.describe()

,user_id,item_id,rating,timestamp
count,100000.00000,100000.000000,100000.000000,1.000000e+05
mean,462.48475,425.530130,3.529860,8.835289e+08
std,266.61442,330.798356,1.125674,5.343856e+06
min,1.00000,1.000000,1.000000,8.747247e+08
25%,254.00000,175.000000,3.000000,8.794487e+08
50%,447.00000,322.000000,4.000000,8.828269e+08
75%,682.00000,631.000000,4.000000,8.882600e+08
max,943.00000,1682.000000,5.000000,8.932866e+08


In [37]:
data.isnull().sum()

user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64

In [38]:
utility_matrix = data.pivot(index='user_id', columns='item_id', values='rating')
user_means = utility_matrix.mean(axis=1)
centered_utility = utility_matrix.sub(user_means, axis=0).fillna(0)
cosine_sim = cosine_similarity(centered_utility)
cosine_sim_df = pd.DataFrame(cosine_sim, index=utility_matrix.index, columns=utility_matrix.index)
cosine_sim_df = pd.DataFrame(cosine_sim, index=utility_matrix.index, columns=utility_matrix.index)

similar_users = cosine_sim_df[1].sort_values(ascending=False)[1:11]  
similar_user_ids = similar_users.index
item_508_ratings = utility_matrix.loc[similar_user_ids, 508].dropna()
if not item_508_ratings.empty:
    expected_rating = item_508_ratings.mean()
    print(f"Expected rating for item 508 for user 1: {expected_rating:.2f}")
else:
    print("No similar users have rated item 508.")

Expected rating for item 508 for user 1: 4.20


## Answer  
Expected rating for item 508 for user 1: 4.20

### Problem 2

In [48]:
# Get users who rated item 95
rated_users = utility_matrix[95].dropna().index

# Get rating vectors for user 15 and user 200
user_15_profile = centered_utility.loc[15]
user_200_profile = centered_utility.loc[200]

# Get a vector of ratings for these users
rated_users_profiles = centered_utility.loc[rated_users]

# Calculate similarities and distances between these users and users 15 and 200
similarity_15 = cosine_similarity([user_15_profile], rated_users_profiles)[0]
similarity_200 = cosine_similarity([user_200_profile], rated_users_profiles)[0]

distance_15 = euclidean_distances([user_15_profile], rated_users_profiles)[0]
distance_200 = euclidean_distances([user_200_profile], rated_users_profiles)[0]

# Get ratings from these users for item 95
ratings_95 = utility_matrix.loc[rated_users, 95]

# Calculate the expected ratings of user 15 and user 200 for item 95
expected_rating_15 = (similarity_15 * ratings_95).sum() / similarity_15.sum()
expected_rating_200 = (similarity_200 * ratings_95).sum() / similarity_200.sum()
for user_id, sim_15, dist_15, sim_200, dist_200 in zip(rated_users, similarity_15, distance_15, similarity_200, distance_200):
    print(f"User {user_id} - Similarity to User 15: {sim_15:.2f}, Distance to User 15: {dist_15:.2f}, Similarity to User 200: {sim_200:.2f}, Distance to User 200: {dist_200:.2f}")
print(f"Expected rating for item 95 for user 15: {expected_rating_15:.2f}")
print(f"Expected rating for item 95 for user 200: {expected_rating_200:.2f}")
# Determine which user to recommend item 95 to
if expected_rating_15 > expected_rating_200:
    print("Recommender system would suggest movie 95 to user 15.")
elif expected_rating_200 > expected_rating_15:
    print("Recommender system would suggest movie 95 to user 200.")
else:
    print("Both users have similar expected ratings for movie 95.")

User 1 - Similarity to User 15: 0.03, Distance to User 15: 24.54, Similarity to User 200: 0.10, Distance to User 200: 23.23
User 5 - Similarity to User 15: 0.03, Distance to User 15: 22.19, Similarity to User 200: 0.07, Distance to User 200: 21.21
User 6 - Similarity to User 15: -0.03, Distance to User 15: 20.62, Similarity to User 200: 0.01, Distance to User 200: 19.52
User 13 - Similarity to User 15: 0.03, Distance to User 15: 37.82, Similarity to User 200: 0.08, Distance to User 200: 36.90
User 16 - Similarity to User 15: -0.02, Distance to User 15: 18.51, Similarity to User 200: 0.08, Distance to User 200: 16.90
User 18 - Similarity to User 15: -0.00, Distance to User 15: 19.62, Similarity to User 200: 0.02, Distance to User 200: 18.73
User 20 - Similarity to User 15: -0.01, Distance to User 15: 15.84, Similarity to User 200: -0.05, Distance to User 200: 15.23
User 23 - Similarity to User 15: 0.07, Distance to User 15: 17.36, Similarity to User 200: 0.07, Distance to User 200: 16.5

## Answer:  
Expected rating for item 95 for user 15: 3.87  
Expected rating for item 95 for user 200: 4.01   
Recommender system would suggest movie 95 to user 200.